In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import Lipinski
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch_geometric.data import Data
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.loader import DataLoader
import warnings
import joblib
import logging
import sys


/home/kamil/dev/ic50-prediction-chembl/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys
from pathlib import Path
project_root = str(Path.cwd().parent)

if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
warnings.filterwarnings("ignore", message=".*The usage of `scatter.*")

In [4]:
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    stream=sys.stdout
)

In [5]:
from src.cleaning.smiles_utils import standardize_smiles
from src.splitting.scaffold_split import scaffold_split
from src.features.graphs import smiles_to_graph_input
from src.training.trainer import run_training
from src.models.gnn import build_gcn_model, build_gine_model

In [6]:
CLEANDED_DATA_LOCATION="../data/raw/batch_0000.parquet"
OUTPUT_LOCATION="../models"

output_dir = Path(OUTPUT_LOCATION)
output_dir.mkdir(parents=True, exist_ok=True)

In [7]:
df = pd.read_parquet(CLEANDED_DATA_LOCATION)
df.head()

,activity_id,molregno,compound_chembl_id,canonical_smiles,standard_type,standard_relation,standard_value,standard_units,pchembl_value,assay_id,assay_type,confidence_score,assay_chembl_id,target_chembl_id,target_name,target_type,organism
0,1655390,2249,CHEMBL7463,CN(C)CCCn1cc(C2=C(c3c[nH]c4ccccc34)C(=O)NC2=O)...,IC50,=,27.0,nM,7.57,326167,B,9,CHEMBL862677,CHEMBL2147,Serine/threonine-protein kinase pim-1,SINGLE PROTEIN,Homo sapiens
1,1655426,3666,CHEMBL50,O=c1c(O)c(-c2ccc(O)c(O)c2)oc2cc(O)cc(O)c12,IC50,=,43.0,nM,7.37,326167,B,9,CHEMBL862677,CHEMBL2147,Serine/threonine-protein kinase pim-1,SINGLE PROTEIN,Homo sapiens
2,1655451,332510,CHEMBL200528,CC(=O)c1cccc(-c2cnc3ccc(NCC4CC4)nn23)c1,IC50,=,61.0,nM,7.21,326167,B,9,CHEMBL862677,CHEMBL2147,Serine/threonine-protein kinase pim-1,SINGLE PROTEIN,Homo sapiens
3,2020259,395940,CHEMBL391586,N#Cc1c(-c2ccccc2)cc(-c2cc(Br)ccc2O)[nH]c1=O,IC50,=,50.0,nM,7.30,454187,B,8,CHEMBL903376,CHEMBL2147,Serine/threonine-protein kinase pim-1,SINGLE PROTEIN,Homo sapiens
4,2020262,408452,CHEMBL247684,N#Cc1c(-c2ccccc2Cl)c2c([nH]c1=O)-c1ccccc1SC2,IC50,=,20000.0,nM,4.70,454187,B,8,CHEMBL903376,CHEMBL2147,Serine/threonine-protein kinase pim-1,SINGLE PROTEIN,Homo sapiens


In [8]:
df = df[['canonical_smiles', 'standard_value']].rename(columns={'canonical_smiles': 'smiles', 'standard_value': 'ic50'})
df.head()

,smiles,ic50
0,CN(C)CCCn1cc(C2=C(c3c[nH]c4ccccc34)C(=O)NC2=O)...,27.0
1,O=c1c(O)c(-c2ccc(O)c(O)c2)oc2cc(O)cc(O)c12,43.0
2,CC(=O)c1cccc(-c2cnc3ccc(NCC4CC4)nn23)c1,61.0
3,N#Cc1c(-c2ccccc2)cc(-c2cc(Br)ccc2O)[nH]c1=O,50.0
4,N#Cc1c(-c2ccccc2Cl)c2c([nH]c1=O)-c1ccccc1SC2,20000.0


In [9]:
df['pic50'] = 9 - np.log10(df['ic50'])

In [10]:
df = df[(df['pic50'] >= 3) & (df['pic50'] <= 12)]
df = df[['smiles', 'pic50']]
df.head()


,smiles,pic50
0,CN(C)CCCn1cc(C2=C(c3c[nH]c4ccccc34)C(=O)NC2=O)...,7.568636
1,O=c1c(O)c(-c2ccc(O)c(O)c2)oc2cc(O)cc(O)c12,7.366532
2,CC(=O)c1cccc(-c2cnc3ccc(NCC4CC4)nn23)c1,7.214670
3,N#Cc1c(-c2ccccc2)cc(-c2cc(Br)ccc2O)[nH]c1=O,7.301030
4,N#Cc1c(-c2ccccc2Cl)c2c([nH]c1=O)-c1ccccc1SC2,4.698970


In [11]:
grouped = df.groupby('smiles')['pic50']
spread = grouped.max() - grouped.min()

smiles_to_drop = spread[spread >= 1].index

df.drop(df[df['smiles'].isin(smiles_to_drop)].index, inplace=True)

df['pic50'] = df.groupby('smiles')['pic50'].transform('median')
df.drop_duplicates(subset=['smiles'], inplace=True)
df.reset_index(drop=True, inplace=True)

In [12]:
df['smiles'] = df['smiles'].apply(standardize_smiles)
df.dropna(subset=['smiles'], inplace=True)
df.reset_index(drop=True, inplace=True)

[19:57:55] Running LargestFragmentChooser
[19:57:55] Running Uncharger
[19:57:55] Running LargestFragmentChooser
[19:57:55] Running Uncharger
[19:57:55] Running LargestFragmentChooser
[19:57:55] Running Uncharger
[19:57:55] Running LargestFragmentChooser
[19:57:55] Running Uncharger
[19:57:55] Running LargestFragmentChooser
[19:57:55] Running Uncharger
[19:57:55] Running LargestFragmentChooser
[19:57:55] Running Uncharger
[19:57:55] Running LargestFragmentChooser
[19:57:55] Running Uncharger
[19:57:55] Running LargestFragmentChooser
[19:57:55] Running Uncharger
[19:57:55] Running LargestFragmentChooser
[19:57:55] Running Uncharger
[19:57:55] Running LargestFragmentChooser
[19:57:55] Running Uncharger
[19:57:55] Running LargestFragmentChooser
[19:57:55] Running Uncharger
[19:57:55] Running LargestFragmentChooser
[19:57:55] Running Uncharger
[19:57:55] Running LargestFragmentChooser
[19:57:55] Running Uncharger
[19:57:55] Running LargestFragmentChooser
[19:57:55] Running Uncharger
[19:57

In [13]:
def compute_descriptors(smiles: str) -> pd.Series:
    mol = Chem.MolFromSmiles(smiles)
    return pd.Series({
        'mol_wt': Descriptors.MolWt(mol),
        'logp': Descriptors.MolLogP(mol),
        'tpsa': Descriptors.TPSA(mol)
    })
df[['mol_wt', 'logp', 'tpsa']] = df['smiles'].apply(compute_descriptors)
df.dropna(subset=['mol_wt', 'logp', 'tpsa'], inplace=True)
df.reset_index(drop=True, inplace=True)

In [14]:
smiles_list = df['smiles'].tolist()
train_idx, val_idx, test_idx = scaffold_split(
    smiles_list=smiles_list, 
    frac_train=0.8, 
    frac_val=0.1, 
    seed=42
)

train_df = df.iloc[train_idx].copy()
val_df = df.iloc[val_idx].copy()
test_df = df.iloc[test_idx].copy()

print(f"Split successful! Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

2026-06-12 19:57:59,135 [INFO] src.splitting.scaffold_split: scaffold_split: train=2444, val=305, test=307 (total=3056)
Split successful! Train: 2444, Val: 305, Test: 307


In [15]:
features_to_scale = ['mol_wt', 'logp', 'tpsa']

feature_scaler = StandardScaler()

train_df[features_to_scale] = feature_scaler.fit_transform(train_df[features_to_scale])

val_df[features_to_scale] = feature_scaler.transform(val_df[features_to_scale])

test_df[features_to_scale] = feature_scaler.transform(test_df[features_to_scale])

In [16]:
joblib.dump(feature_scaler, f"{OUTPUT_LOCATION}/global_features_scaler.pkl")

['../models/global_features_scaler.pkl']

In [17]:
def build_dataset(df):
    dataset = []
    
    for _, row in df.iterrows():
        smiles = row['smiles']
        pic50 = row['pic50']
        
        graph_data = smiles_to_graph_input(smiles, feature_scaler)
        graph_data.y = torch.tensor([pic50], dtype=torch.float)
        
        if graph_data is not None:
            dataset.append(graph_data)
            
    return dataset

In [18]:
train_graphs = build_dataset(train_df)
val_graphs = build_dataset(val_df)
test_graphs = build_dataset(test_df)

/home/kamil/dev/ic50-prediction-chembl/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/kamil/dev/ic50-prediction-chembl/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/kamil/dev/ic50-prediction-chembl/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/kamil/dev/ic50-prediction-chembl/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/kamil/dev/ic50-prediction-chembl/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:26

In [22]:
def execute_training_pipeline(
    train_graphs: list,
    val_graphs: list,
    test_graphs: list,
    output_location: str | Path,
    model_type: str = "gcn",  # Flag to toggle the model
    batch_size: int = 128,
    max_epochs: int = 200,
    early_stopping_patience: int = 50,
    lr: float = 1e-3
) -> dict:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_loader = DataLoader(train_graphs, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_graphs, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_graphs, batch_size=batch_size, shuffle=False)

    if model_type.lower() == "gcn":
        model = build_gcn_model()
        save_filename = "best_gcn.pt"
    elif model_type.lower() == "gine":
        model = build_gine_model()
        save_filename = "best_gine.pt"
    else:
        raise ValueError(f"Invalid model_type: '{model_type}'. Choose 'gine' or 'gcn'.")
    
    model = model.to(device)

    optimizer = Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    loss_fn = nn.MSELoss()

    lr_scheduler = ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.8,
        patience=40,
        min_lr=1e-6,
    )

    output_dir = Path(output_location)
    output_dir.mkdir(parents=True, exist_ok=True)
    model_save_path = output_dir / save_filename

    print(f"Starting training for {model_type.upper()} model...")
    
    training_history = run_training(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        optimizer=optimizer,
        lr_scheduler=lr_scheduler,
        loss_fn=loss_fn,
        max_epochs=max_epochs,
        patience=early_stopping_patience,
        model_save_path=model_save_path,
        device=device
    )

    return training_history

In [20]:
gcn_history = execute_training_pipeline(
    train_graphs=train_graphs,
    val_graphs=val_graphs,
    test_graphs=test_graphs,
    output_location=OUTPUT_LOCATION,
    model_type="gcn"
)


Starting training for GCN model...
2026-06-12 19:58:03,263 [INFO] src.training.trainer: Training on device: cuda


2026-06-12 19:58:04,378 [INFO] src.training.trainer: Epoch 001 | train_loss=42.4937 | val RMSE: 4.9372 | R²: -14.7550 | MAE: 4.7809 | LR: 0.001000  *


2026-06-12 19:58:05,193 [INFO] src.training.trainer: Epoch 002 | train_loss=7.6641 | val RMSE: 1.4292 | R²: -0.3203 | MAE: 1.1217 | LR: 0.001000  *


2026-06-12 19:58:06,025 [INFO] src.training.trainer: Epoch 003 | train_loss=1.6997 | val RMSE: 1.2881 | R²: -0.0724 | MAE: 1.0118 | LR: 0.001000  *


2026-06-12 19:58:06,886 [INFO] src.training.trainer: Epoch 004 | train_loss=1.3000 | val RMSE: 1.0588 | R²: 0.2754 | MAE: 0.8441 | LR: 0.001000  *


...

2026-06-12 19:59:12,510 [INFO] src.training.trainer: Epoch 085 | train_loss=0.7084 | val RMSE: 0.9398 | R²: 0.4292 | MAE: 0.7135 | LR: 0.000410 


2026-06-12 19:59:13,348 [INFO] src.training.trainer: Epoch 086 | train_loss=0.7881 | val RMSE: 1.0156 | R²: 0.3333 | MAE: 0.7804 | LR: 0.000410 


2026-06-12 19:59:14,176 [INFO] src.training.trainer: Epoch 087 | train_loss=0.7067 | val RMSE: 1.0475 | R²: 0.2909 | MAE: 0.8058 | LR: 0.000410 


2026-06-12 19:59:14,993 [INFO] src.training.trainer: Epoch 088 | train_loss=0.7326 | val RMSE: 0.9231 | R²: 0.4493 | MAE: 0.7000 | LR: 0.000410 
2026-06-12 19:59:14,994 [INFO] src.training.trainer: Early stopping after 25 epochs without improvement.
2026-06-12 19:59:14,994 [INFO] src.training.trainer: Training complete. Best val RMSE=0.8359 at epoch 63.


2026-06-12 19:59:15,307 [INFO] src.training.trainer: Test metrics: RMSE: 0.7982 | R²: 0.5509 | MAE: 0.6353


In [23]:
gnn_history = execute_training_pipeline(
    train_graphs=train_graphs,
    val_graphs=val_graphs,
    test_graphs=test_graphs,
    output_location=OUTPUT_LOCATION,
    model_type="gine"
)

Starting training for GINE model...
2026-06-12 20:00:40,644 [INFO] src.training.trainer: Training on device: cuda


  train:   0%|          | 0/20 [00:00<?, ?batch/s]

2026-06-12 20:00:41,325 [INFO] src.training.trainer: Epoch 001 | train_loss=18.7182 | val RMSE: 4.7948 | R²: -13.8593 | MAE: 4.6341 | LR: 0.001000  *


2026-06-12 20:00:42,098 [INFO] src.training.trainer: Epoch 002 | train_loss=2.5673 | val RMSE: 1.4666 | R²: -0.3901 | MAE: 1.2530 | LR: 0.001000  *


2026-06-12 20:00:42,868 [INFO] src.training.trainer: Epoch 003 | train_loss=2.1343 | val RMSE: 0.9980 | R²: 0.3563 | MAE: 0.7770 | LR: 0.001000  *


...

2026-06-12 20:02:54,437 [INFO] src.training.trainer: Epoch 173 | train_loss=1.0413 | val RMSE: 0.7746 | R²: 0.6122 | MAE: 0.5934 | LR: 0.000800 


2026-06-12 20:02:55,240 [INFO] src.training.trainer: Epoch 174 | train_loss=0.9865 | val RMSE: 0.7512 | R²: 0.6353 | MAE: 0.5816 | LR: 0.000800 


2026-06-12 20:02:56,018 [INFO] src.training.trainer: Epoch 175 | train_loss=1.0229 | val RMSE: 0.7924 | R²: 0.5942 | MAE: 0.5932 | LR: 0.000800 


2026-06-12 20:02:56,794 [INFO] src.training.trainer: Epoch 176 | train_loss=0.9930 | val RMSE: 0.7585 | R²: 0.6282 | MAE: 0.5958 | LR: 0.000800 
2026-06-12 20:02:56,794 [INFO] src.training.trainer: Early stopping after 50 epochs without improvement.
2026-06-12 20:02:56,795 [INFO] src.training.trainer: Training complete. Best val RMSE=0.7375 at epoch 126.


2026-06-12 20:02:57,082 [INFO] src.training.trainer: Test metrics: RMSE: 0.7190 | R²: 0.6356 | MAE: 0.5796
